# Parameter optimization of the denoising algorithm

## 1. Import libraries

In [14]:
import numpy as np
import pandas as pd

from src.evaluation.evaluation import read_results
from src.optimization.splits import load_split, split_results

pd.set_option('display.max_columns', 500)

## 2. Score definition

### 2.1. Per-artifact quality

$$
q_i = \frac{\text{clip}(\text{SNR}_i^{\text{out}}, F, C) - F}{C-F}
$$

where $F=-30$ dB and $C=0$ dB.

### 2.2. Global quality

$$
Q = \frac{1}{|\mathcal{S}|}\sum_{s\in\mathcal{S}} Q_s, \qquad
Q_s = \frac{1}{|\mathcal{A}| |\mathcal{C}|}\sum_{a\in\mathcal{A}}\sum_{c\in\mathcal{C}} \bar{q}_{s,a,c}
$$

- $Q_s$: quality at SNR $s$.
- $\mathcal{S}$, $\mathcal{A}$, $\mathcal{C}$: sets of SNR values, artifacts and clean signals (every artifact is inserted into every clean signal).
- $\bar{q}_{s,a,c}$: mean quality of artifact $a$ in clean signal $c$ at SNR $s$ (mean over its $N_{a,c}$ positions):

$$
\bar{q}_{s,a,c} = \frac{1}{N_{a,c}}\sum_{i=1}^{N_{a,c}} q_{s,a,c}^{(i)}
$$

> Each artifact and each clean signal weighs the same, whatever its number of positions $N_{a,c}$.

### 2.3. Number of false positives

$$
FP = \frac{1}{|\mathcal{C}|}\sum_{c\in\mathcal{C}}\bar{n}_c^{\text{false}}, \qquad
\bar{n}_c^{\text{false}} = \frac{1}{|\mathcal{S}| |\mathcal{A}|}\sum_{s\in\mathcal{S}}\sum_{a\in\mathcal{A}} \bar{n}_{s,a,c}^{\text{false}}
$$

- $\bar{n}_c^{\text{false}}$: mean number of false positives of the mixtures of clean signal $c$, averaged as $Q$.
- $\bar{n}_{s,a,c}^{\text{false}}$: mean over the $N_{a,c}$ positions.

### 2.4. Score

$$
S = Q - \lambda\cdot FP
$$

- $\lambda$: penalty of the false positives (see [3.2](#3.2.-Penalty-λ)).

### 2.5. Diagnostics

- Detection per SNR.
- SNR at 50 % detection.
- Conservation.
- False positives per label.
- Local RMS error of the detected artifacts.

In [15]:
F_DB = -30  # dB. Per-artifact quality: q = 0 at SNR_out <= F_DB ...
C_DB = 0    # dB. ... and q = 1 at SNR_out >= C_DB


def quality(snr_out_db, f=F_DB, c=C_DB):
    """
    Per-artifact quality: q = (clip(SNR_out, F, C) - F) / (C - F).

    Parameters
    ----------
    snr_out_db: array-like
        SNR in the artifact zone after the filter (dB)
    f, c: float
        Floor and ceiling of the SNR (dB): q = 0 at SNR_out <= f, q = 1 at SNR_out >= c

    Returns
    -------
    q: array-like
        Quality of each artifact, in [0, 1]
    """
    return (np.clip(snr_out_db, f, c) - f) / (c - f)


def snr_at_50(rate, levels):
    """
    SNR at which the detection rate crosses 0.5 (linear interpolation between levels), as in plot_detection of
    evaluation_denoise.ipynb. The detection decreases as the SNR increases: the last level with rate >= 0.5 is taken
    and interpolated up to the next one.
    +inf: >= 0.5 up to the last level (crossing above the range); -inf: < 0.5 in all of them (below the range).

    Parameters
    ----------
    rate: np.ndarray
        Detection rate at each SNR level
    levels: np.ndarray
        SNR levels (dB), in increasing order

    Returns
    -------
    snr50: float
        SNR at 50 % detection (dB)
    """
    above = rate >= 0.5
    if not above.any():
        return -np.inf
    i = np.where(above)[0].max()
    if i == len(rate) - 1:
        return np.inf
    r0, r1 = rate[i], rate[i + 1]
    return levels[i] + (0.5 - r0) * (levels[i + 1] - levels[i]) / (r1 - r0)


def score(results_df, lam, snr_list=None, f=F_DB, c=C_DB):
    """
    Score S = Q - λ·FP of the results of an experiment with one artifact per mixture (format of experiment_1.csv),
    and its diagnostics (see section 2). Every artifact is inserted into every clean signal, and the means are taken
    first over the positions of each cell (s, a, c), so that each artifact and each clean signal weighs the same
    whatever its number of positions:
    - Q: mean over the positions -> the clean signals -> the artifacts -> the SNR of the quality
    - FP: for each clean signal, mean over the positions -> the artifacts -> the SNR of the number of false positives
      of its mixtures (controls excluded); then mean over the clean signals

    Parameters
    ----------
    results_df: pd.DataFrame
        Results (mixtures + controls), already restricted to the set to evaluate (e.g. tuning signals and artifacts)
    lam: float
        Penalty λ of the false positives
    snr_list: list or None
        SNR values (dB) of the set S. None -> all the SNR of the mixtures. FP uses the mixtures of the same SNR values
    f, c: float
        Floor and ceiling of the per-artifact quality (dB)

    Returns
    -------
    scores: dict
        S, Q, FP and lam
    diagnostics: dict
        - per_snr: pd.DataFrame, one row per SNR: number of mixtures, detection rate Det_s and quality Q_s (averaged
          as Q), mean q of the detected / undetected artifacts and dq = q_det - q_undet (cost of missing an artifact,
          see 3.2), median NMSE out of the artifact zone and median local RMS error (max, mean) after the filter of the
          detected artifacts.
          q_det and q_undet are weighted as in Q (each mixture weighs w_i = 1 / N_{a,c}, number of positions of its
          cell), so that Q_s = Det_s·q_det + (1 - Det_s)·q_undet exactly (see 6.3)
        - snr50: SNR at 50 % detection of all the artifacts (dB; ±inf: crossing out of the SNR range)
        - snr50_art: pd.Series, SNR at 50 % detection of each artifact
        - fp_label: pd.DataFrame, false positives per label of the clean signal: mean over its clean signals of the
          mean per mixture (as FP), mean per control and number of clean signals
        - nmse_control_db: median NMSE of the controls (whole signal)
    """
    df = results_df.assign(snr_db=pd.to_numeric(results_df["snr_db"], errors="coerce"))  # "Control" -> NaN
    is_control = df["art_id"].astype(str) == "Control"
    controls = df[is_control]
    mix = df[~is_control]
    if snr_list is not None:
        mix = mix[mix["snr_db"].isin(snr_list)]
    mix = mix.assign(q=quality(mix["snr_out_db"], f, c), detected=mix["detected"].astype(bool))

    cells = ["snr_db", "art_id", "clean_id"]

    def per_snr_mean(cell_values):
        """Mean of the values of the cells (s, a, c) over the clean signals, then over the artifacts -> one per SNR."""
        return cell_values.groupby(level=["snr_db", "art_id"]).mean().groupby(level="snr_db").mean()

    # ============================================================================ score
    cell_q = mix.groupby(cells)["q"].mean()                   # q̄_{s,a,c}: mean over the positions
    q_snr = per_snr_mean(cell_q)                              # Q_s: mean over the clean signals, then the artifacts
    Q = q_snr.mean()                                          # Q: mean over the SNR
    fp_clean = (mix.groupby(["clean_id", "snr_db", "art_id"])["n_false"].mean()   # mean over the positions
                .groupby(level=["clean_id", "snr_db"]).mean()                     # mean over the artifacts
                .groupby(level="clean_id").mean())                                # n̄_c^false: mean over the SNR
    FP = fp_clean.mean()
    S = Q - lam * FP
    scores = dict(S=S, Q=Q, FP=FP, lam=lam)

    # ============================================================================ diagnostics
    # Detection, quality and conservation per SNR
    cell_det = mix.groupby(cells)["detected"].mean()          # p_k: fraction of positions detected of each cell
    # Weight of each mixture in Q_s: w_i = 1 / N_{a,c} (number of positions of its cell)
    mix = mix.assign(w=1 / mix.groupby(cells)["q"].transform("size"))
    det, undet = mix[mix["detected"]], mix[~mix["detected"]]

    def weighted_q(d):
        """Mean q per SNR of the mixtures d, each one weighted by w."""
        return (d["w"] * d["q"]).groupby(d["snr_db"]).sum() / d.groupby("snr_db")["w"].sum()

    per_snr = pd.DataFrame({
        "n_mixtures": mix.groupby("snr_db").size(),
        "detection": per_snr_mean(cell_det),                  # Det_s
        "q": q_snr,                                           # Q_s
        "q_det": weighted_q(det),                             # q̄_s^det
        "q_undet": weighted_q(undet),                         # q̄_s^undet
        "nmse_out_zone_db": mix.groupby("snr_db")["nmse_out_zone_db"].median(),
        "err_rms_max_loc_out": det.groupby("snr_db")["err_rms_max_loc_out"].median(),
        "err_rms_mean_loc_out": det.groupby("snr_db")["err_rms_mean_loc_out"].median(),
    })
    per_snr.insert(per_snr.columns.get_loc("q_undet") + 1, "dq", per_snr["q_det"] - per_snr["q_undet"])
    per_snr.index.name = "snr_db"

    # SNR at 50 % detection: all the artifacts and each artifact (detection rate = mean over the clean signals)
    levels = per_snr.index.to_numpy()
    snr50 = snr_at_50(per_snr["detection"].to_numpy(), levels)
    rate_art = cell_det.groupby(level=["art_id", "snr_db"]).mean().unstack("snr_db").reindex(columns=levels)
    snr50_art = rate_art.apply(lambda r: snr_at_50(r.to_numpy(), levels), axis=1).rename("snr50_db")

    # False positives per label of the clean signal
    label = df.groupby("clean_id")["label"].first()
    fp_label = pd.DataFrame({
        "fp_mixture": fp_clean.groupby(label).mean(),
        "fp_control": controls.groupby("label")["n_false"].mean(),
        "n_clean": fp_clean.groupby(label).size(),
    })
    fp_label.index.name = "label"

    diagnostics = dict(per_snr=per_snr, snr50=snr50, snr50_art=snr50_art, fp_label=fp_label,
                       nmse_control_db=controls["nmse_out_zone_db"].median())

    return scores, diagnostics

In [16]:
def describe_score(scores, diagnostics, artifact_info=None, n_worst=5):
    """
    Descriptive summary of the Score of an experiment (output of score), printed and displayed:
    - Score: S, Q, FP and weight of the penalty λ·FP in the total loss 1 - S = (1 - Q) + λ·FP
    - Quality lost per SNR, 1 - Q_s, split into the missed artifacts, (1 - Det_s)·(1 - q̄_s^undet), and the imperfect
      attenuation of the detected ones, Det_s·(1 - q̄_s^det)
    - Conservation (NMSE out of the artifact zone) and local RMS error of the detected artifacts per SNR
    - Detection: SNR at 50 % detection of all the artifacts and of each one (per duration, if artifact_info is
      given, and hardest artifacts)
    - False positives per label and median NMSE of the controls

    Parameters
    ----------
    scores, diagnostics: dict
        Output of score
    artifact_info: pd.DataFrame or None
        Table with the columns art_id and art_duration_ms (e.g. artifact_split). None -> no durations
    n_worst: int
        Number of hardest artifacts (lowest SNR at 50 % detection) shown
    """
    S, Q, FP, lam = scores["S"], scores["Q"], scores["FP"], scores["lam"]
    per_snr = diagnostics["per_snr"]

    # ============================================================================ score
    print("Score")
    print(f"  S = Q - λ·FP = {S:.4f}   (λ = {lam})")
    print(f"  Q = {Q:.4f}   (quality lost: 1 - Q = {1 - Q:.4f})")
    print(f"  FP = {FP:.4f} false positives per mixture   (penalty: λ·FP = {lam * FP:.4f}, "
          f"{lam * FP / (1 - S):.1%} of the total loss 1 - S = {1 - S:.4f})")

    # ============================================================================ quality lost per SNR
    # 1 - Q_s = (1 - Det_s)·(1 - q̄_s^undet) + Det_s·(1 - q̄_s^det); NaN (no undetected / detected mixtures) -> 0
    loss = pd.DataFrame({
        "detection": per_snr["detection"],
        "q": per_snr["q"],
        "q_det": per_snr["q_det"],
        "q_undet": per_snr["q_undet"],
        "dq": per_snr["dq"],
        "loss": 1 - per_snr["q"],
        "loss_missed": (1 - per_snr["detection"]) * (1 - per_snr["q_undet"].fillna(1)),
        "loss_attenuation": per_snr["detection"] * (1 - per_snr["q_det"].fillna(1)),
    })
    mean_loss = loss[["loss", "loss_missed", "loss_attenuation"]].mean()     # mean over the SNR: 1 - Q
    print(f"\nQuality lost per SNR: 1 - Q = {mean_loss['loss']:.4f} = "
          f"{mean_loss['loss_missed']:.4f} missed artifacts ({mean_loss['loss_missed'] / mean_loss['loss']:.1%}) + "
          f"{mean_loss['loss_attenuation']:.4f} imperfect attenuation "
          f"({mean_loss['loss_attenuation'] / mean_loss['loss']:.1%})")
    display(loss.round(4))

    # ============================================================================ conservation and local error
    print("\nConservation and local RMS error of the detected artifacts per SNR (medians)")
    display(per_snr[["n_mixtures", "nmse_out_zone_db", "err_rms_max_loc_out", "err_rms_mean_loc_out"]].round(3))

    # ============================================================================ detection
    snr50_art = diagnostics["snr50_art"].copy()
    snr50_art.index = pd.to_numeric(snr50_art.index)
    print(f"\nSNR at 50 % detection: {diagnostics['snr50']:.1f} dB (all the artifacts)")
    print(f"  Per artifact: median {snr50_art.median():.1f} dB, range [{snr50_art.min():.1f}, {snr50_art.max():.1f}] dB")
    art = snr50_art.to_frame()
    if artifact_info is not None:
        durations = (artifact_info.assign(art_id=pd.to_numeric(artifact_info["art_id"]))
                     .set_index("art_id")["art_duration_ms"])
        art = art.join(durations)
        print("  Per duration of the artifact (ms): median SNR at 50 % detection and number of artifacts")
        display(art.groupby("art_duration_ms")["snr50_db"].agg(["median", "size"]).round(1))
    print("  Hardest artifacts (lowest SNR at 50 % detection)")
    display(art.sort_values("snr50_db").head(n_worst).round(1))

    # ============================================================================ false positives
    print(f"\nFalse positives per label (median NMSE of the controls: {diagnostics['nmse_control_db']:.1f} dB)")
    display(diagnostics["fp_label"].round(3))

## 3. Phase 0: Preparation

- **Output:** tuning/test split, $\lambda$ and baseline.

### 3.1. Tuning/test split

75 % tuning / 25 % test (first of 4 folds as test, [get_splits.py](get_splits.py)), grouped by signal and artifact:

- **Signals:** stratified by label.
- **Artifacts:** stratified by duration.

| Set | Tuning | Test | Total |
|:----|:------:|:----:|:-----:|
| **Signals**   | 29 | 9 | 38 |
| **Artifacts** | 23 | 9 | 32 |

Signal × artifact combinations: **tuning** (29 × 23), **test** (9 × 9), **new signals** (9 × 23) and **new artifacts** (29 × 9).

In [17]:
clean_split, artifact_split = load_split()  # denoising/splits (generated by get_splits.py)

tune_clean = set(clean_split.loc[clean_split["set"] == "tune", "clean_id"])
test_clean = set(clean_split.loc[clean_split["set"] == "test", "clean_id"])
tune_art = set(artifact_split.loc[artifact_split["set"] == "tune", "art_id"])
test_art = set(artifact_split.loc[artifact_split["set"] == "test", "art_id"])

# Clean signals per label and set
signals = pd.crosstab(clean_split["label"], clean_split["set"], margins=True, margins_name="Total")
signals = signals[["tune", "test", "Total"]]
print("Clean signals per label")
display(signals)

# Artifacts per duration and set
artifacts = pd.crosstab(artifact_split["art_duration_ms"], artifact_split["set"], margins=True, margins_name="Total")
artifacts = artifacts[["tune", "test", "Total"]]
print("Artifacts per duration (ms)")
display(artifacts)

# Signal x artifact combinations of each set
combos = pd.DataFrame({
    "signals": [len(tune_clean), len(test_clean), len(test_clean), len(tune_clean)],
    "artifacts": [len(tune_art), len(test_art), len(tune_art), len(test_art)],
}, index=pd.Index(["tune", "test", "new_signals", "new_artifacts"], name="set"))
combos["combinations"] = combos["signals"] * combos["artifacts"]
print("Signal x artifact combinations")
display(combos)

# Ids of each set
print(f"Tuning clean signals: {sorted(tune_clean)}")
print(f"Test clean signals:   {sorted(test_clean)}")
print(f"Tuning artifacts:     {sorted(tune_art)}")
print(f"Test artifacts:       {sorted(test_art)}")

Clean signals per label


set,tune,test,Total
label,,,
Normal,7,3,10
Rhonchi,8,2,10
Stridor,6,2,8
Wheeze,8,2,10
Total,29,9,38


Artifacts per duration (ms)


set,tune,test,Total
art_duration_ms,,,
30.0,3,2,5
40.0,12,4,16
50.0,6,1,7
60.0,1,0,1
70.0,0,1,1
80.0,1,0,1
90.0,0,1,1
Total,23,9,32


Signal x artifact combinations


,signals,artifacts,combinations
set,,,
tune,29,23,667
test,9,9,81
new_signals,9,23,207
new_artifacts,29,9,261


Tuning clean signals: [0, 1, 3, 4, 5, 6, 8, 10, 12, 13, 14, 16, 17, 18, 19, 20, 21, 23, 24, 25, 27, 28, 29, 30, 31, 32, 35, 36, 37]
Test clean signals:   [2, 7, 9, 11, 15, 22, 26, 33, 34]
Tuning artifacts:     [0, 1, 2, 3, 5, 7, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 27, 28]
Test artifacts:       [4, 6, 8, 13, 20, 26, 29, 30, 31]


In [18]:
results_df = read_results("experiment_1.csv")

# Split into the four sets of section 3.1: each one keeps the controls of its clean signals
sets = split_results(results_df, clean_split, artifact_split)
is_control = results_df["art_id"].astype(str) == "Control"

# Summary of each set
summary = {}
for name, df in sets.items():
    ctrl = df["art_id"].astype(str) == "Control"
    summary[name] = {
        "signals": df["clean_id"].nunique(),
        "artifacts": df.loc[~ctrl, "art_id"].nunique(),
        "combinations": df[~ctrl].groupby(["clean_id", "art_id"]).ngroups,
        "mixtures": (~ctrl).sum(),
        "controls": ctrl.sum(),
    }
summary = pd.DataFrame(summary).T
summary.index.name = "set"
display(summary)

tune = sets["tune"]
test = sets["test"]
new_signals = sets["new_signals"]
new_artifacts = sets["new_artifacts"]

,signals,artifacts,combinations,mixtures,controls
set,,,,,
tune,29,23,667,26664,29
test,9,9,81,3240,9
new_signals,9,23,207,8280,9
new_artifacts,29,9,261,10432,29


In [5]:
tune

,id,clean_id,clean_file,label,art_id,rep,snr_db,k0,k1,t0_s,art_duration_ms,g,p_local_rel_db,snr_global_db,detected,n_false,snr_in_db,snr_out_db,dsnr_db,attenuated,nmse_out_zone_db,err_rms_max_loc_mix,err_rms_max_loc_out,err_rms_mean_loc_mix,err_rms_mean_loc_out,err_rms_max_glob_mix,err_rms_max_glob_out,err_rms_mean_glob_mix,err_rms_mean_glob_out
0,0,0,19,Normal,0,0.0,-30,2224.0,2344.0,0.556,30.0,11.657671,1.929284,-12.912431,True,0,-30.000162,-2.293803,2.770636e+01,True,-48.357410,2276.685910,-18.708341,1362.163855,-11.608118,1469.912947,-5.463990e-05,63.351784,-5.375969e-01
1,1,0,19,Normal,0,0.0,-20,2224.0,2344.0,0.556,30.0,3.686479,1.929284,-2.912431,True,0,-20.000162,-1.878164,1.812200e+01,True,-49.934976,649.297471,-13.237228,388.410368,-4.462894,394.946259,-1.727932e-05,18.064209,-2.142408e-01
2,2,0,19,Normal,0,0.0,-10,2224.0,2344.0,0.556,30.0,1.165767,1.929284,7.087569,False,0,-10.000162,-10.000162,0.000000e+00,False,-50.061333,143.788973,143.788973,92.022027,92.022027,61.034095,6.103410e+01,4.279768,4.273706e+00
3,3,0,19,Normal,0,0.0,-5,2224.0,2344.0,0.556,30.0,0.655559,1.929284,12.087569,False,0,-5.000162,-5.000162,0.000000e+00,False,-50.076448,48.527876,48.527876,38.867207,38.867207,-0.000003,-3.072792e-06,1.807643,1.801581e+00
4,4,0,19,Normal,0,0.0,0,2224.0,2344.0,0.556,30.0,0.368648,1.929284,17.087569,False,0,-0.000162,-0.000162,-9.643707e-16,False,-50.081239,8.329665,8.329665,14.160422,14.160422,-0.000002,-1.727960e-06,0.658579,6.525167e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48647,48671,31,9423,Stridor,Control,NaN,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,-59.446377,NaN,NaN,NaN,NaN,0.000000,1.807245e-03,0.000000,-4.734672e-04
48648,48672,32,9735,Stridor,Control,NaN,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,-19.918233,NaN,NaN,NaN,NaN,0.000000,-1.209588e-14,0.000000,-5.146678e+00
48651,48675,35,17812,Stridor,Control,NaN,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,-74.983649,NaN,NaN,NaN,NaN,0.000000,-1.308418e-14,0.000000,4.221891e-05
48652,48676,36,23218,Stridor,Control,NaN,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,-312.018659,NaN,NaN,NaN,NaN,0.000000,-1.293224e-14,0.000000,-1.884035e-13


### 3.2. Penalty λ

One false positive is worth missing an artifact right at the masking limit $s^*$.

**Masking SNR $s^*$:**

- **Masking:** the artifact raises the global RMS peak (moving RMS, 30 ms) by more than 10 % (`err_rms_max_glob_mix` > 10 %).
- $s^*$ = SNR at which 50 % of the mixtures are masked (linear interpolation between the SNR of `SNR_LIST`).

**Cost of missing an artifact at $s^*$**, computed in two ways:

1. **Approximation (ideal elimination, $q^{\text{det}} = 1$):**

$$
\lambda \approx -\frac{s^*}{30}
$$

2. **Measured** ($q^{\text{det}} < 1$):

$$
\Delta q_s = \bar{q}^{\text{det}}_s - \bar{q}^{\text{undet}}_s, \qquad \lambda = \Delta q_{s^*}
$$

- $\bar{q}^{\text{det}}_s$ / $\bar{q}^{\text{undet}}_s$: mean $q$ of the detected / undetected artifacts at SNR $s$, averaged as in $Q$ (see [Decomposition of the quality](#Decomposition-of-the-quality) in 6.3).
- $\Delta q_{s^*}$: interpolated linearly between the SNR of `SNR_LIST`.
- Computed with the current results (`experiment_1.csv`), on the tuning set only.

In [19]:
F_DB = -30  # dB. Per-artifact quality: q = 0 at SNR_out <= F_DB ...
C_DB = 0    # dB. ... and q = 1 at SNR_out >= C_DB

tune = tune.assign(snr_db=pd.to_numeric(results_df["snr_db"], errors="coerce"))
is_control = tune["art_id"].astype(str) == "Control"
mix = tune[~is_control]

# Masked mixtures
mix["masked"] = mix["err_rms_max_glob_mix"] > 10

# Masking by SNR
cells = ["snr_db", "art_id", "clean_id"]
masked_snr = mix.groupby(cells)["masked"].mean().groupby(["snr_db", "art_id"]).mean().groupby("snr_db").mean()
print("Masked rate by SNR:")
print(masked_snr)

# SNR at 50% of masking
y = masked_snr.values
x = masked_snr.index
above = y >= 0.5
i = np.where(above)[0].max()
y0, y1 = y[i], y[i+1]
x0, x1 = x[i], x[i+1]
s_star = x0 + (0.5 - y0) * (x1 - x0) / (y1 - y0)
print(f"s* = {s_star}\n")
print(f"λ approximated: λ = -s*/30 = {-s_star/30}")

# Quality
mix["q"] = (np.clip(mix["snr_out_db"], F_DB, C_DB) - F_DB) / (C_DB - F_DB)

# q̄_s^det / q̄_s^undet: mean q of the detected / undetected mixtures at each SNR, weighted as in Q: w_i = 1 / N_{a,c}
mix["detected"] = mix["detected"].astype(bool)
mix["w"] = 1 / mix.groupby(cells)["q"].transform("size")   # N_{a,c}: number of positions of the cell of mixture i
det, undet = mix[mix["detected"]], mix[~mix["detected"]]
q_det_snr = (det["w"] * det["q"]).groupby(det["snr_db"]).sum() / det.groupby("snr_db")["w"].sum()
q_undet_snr = (undet["w"] * undet["q"]).groupby(undet["snr_db"]).sum() / undet.groupby("snr_db")["w"].sum()

q_snr = pd.DataFrame({
    "q_det": q_det_snr,                                                      # q̄_s^det
    "q_undet": q_undet_snr,                                                  # q̄_s^undet
})
q_snr["dq"] = q_snr["q_det"] - q_snr["q_undet"]                             # Δq_s
print("\nMean q of the detected / undetected artifacts by SNR:")
display(q_snr)

# Δq at s*
y = q_snr["dq"].values
x = q_snr.index
above = x >= s_star
i = np.where(above)[0].min()
y0, y1 = y[i], y[i-1]
x0, x1 = x[i], x[i-1]
lam = y0 + (y1 - y0)/(x1 - x0) * (s_star - x0)
print(f"λ measured: λ = Δq_s* = {lam}")

Masked rate by SNR:
snr_db
-30.0    0.945427
-20.0    0.840180
-10.0    0.602399
-5.0     0.371289
 0.0     0.166417
 5.0     0.061019
 10.0    0.014468
 20.0    0.000000
Name: masked, dtype: float64
s* = -7.784625364904313

λ approximated: λ = -s*/30 = 0.2594875121634771

Mean q of the detected / undetected artifacts by SNR:


,q_det,q_undet,dq
snr_db,,,
-30.0,0.781984,0.000017,0.781967
-20.0,0.872830,0.333337,0.539492
-10.0,0.936910,0.666673,0.270236
-5.0,0.969303,0.833341,0.135962
0.0,0.992741,0.999991,-0.007250
5.0,0.990227,1.000000,-0.009773
10.0,0.986202,1.000000,-0.013798
20.0,0.981755,1.000000,-0.018245


λ measured: λ = Δq_s* = 0.21074258699499776


### 3.3. Baseline

Score, Q and FP of the current version on the tuning set, with the $\lambda$ of [3.2](#3.2.-Penalty-λ).

In [22]:
lam = 0.21  # λ measured in 3.2: Δq at s* = -7.8 dB (0.2107); approximation -s*/30 = 0.26

scores, diagnostics = score(tune, lam)
describe_score(scores, diagnostics, artifact_info=artifact_split)

Score
  S = Q - λ·FP = 0.8987   (λ = 0.21)
  Q = 0.9176   (quality lost: 1 - Q = 0.0824)
  FP = 0.0900 false positives per mixture   (penalty: λ·FP = 0.0189, 18.7% of the total loss 1 - S = 0.1013)

Quality lost per SNR: 1 - Q = 0.0824 = 0.0335 missed artifacts (40.7%) + 0.0489 imperfect attenuation (59.3%)


,detection,q,q_det,q_undet,dq,loss,loss_missed,loss_attenuation
snr_db,,,,,,,,
-30.0,0.9898,0.7740,0.7820,0.0000,0.7820,0.2260,0.0102,0.2158
-20.0,0.9391,0.8400,0.8728,0.3333,0.5395,0.1600,0.0406,0.1194
-10.0,0.6646,0.8463,0.9369,0.6667,0.2702,0.1537,0.1118,0.0419
-5.0,0.3656,0.8830,0.9693,0.8333,0.1360,0.1170,0.1057,0.0112
0.0,0.1471,0.9989,0.9927,1.0000,-0.0072,0.0011,0.0000,0.0011
5.0,0.0642,0.9994,0.9902,1.0000,-0.0098,0.0006,0.0000,0.0006
10.0,0.0378,0.9995,0.9862,1.0000,-0.0138,0.0005,0.0000,0.0005
20.0,0.0270,0.9995,0.9818,1.0000,-0.0182,0.0005,0.0000,0.0005



Conservation and local RMS error of the detected artifacts per SNR (medians)


,n_mixtures,nmse_out_zone_db,err_rms_max_loc_out,err_rms_mean_loc_out
snr_db,,,,
-30.0,3333,-40.223,-3.125,-4.849
-20.0,3333,-44.965,-0.999,-1.986
-10.0,3333,-50.268,-0.026,0.395
-5.0,3333,-52.052,-0.067,-0.279
0.0,3333,-53.380,-6.256,-5.765
5.0,3333,-53.621,-18.110,-17.690
10.0,3333,-53.630,-33.273,-26.130
20.0,3333,-53.622,-58.284,-33.627



SNR at 50 % detection: -7.2 dB (all the artifacts)
  Per artifact: median -7.2 dB, range [-11.5, -1.2] dB
  Per duration of the artifact (ms): median SNR at 50 % detection and number of artifacts


,median,size
art_duration_ms,,
30.0,-8.4,3
40.0,-7.3,12
50.0,-7.8,6
60.0,-6.8,1
80.0,-7.1,1


  Hardest artifacts (lowest SNR at 50 % detection)


,snr50_db,art_duration_ms
art_id,,
0,-11.5,30.0
10,-10.1,50.0
17,-9.8,40.0
24,-9.8,40.0
14,-9.0,50.0



False positives per label (median NMSE of the controls: -53.8 dB)


,fp_mixture,fp_control,n_clean
label,,,
Normal,0.000,0.000,7
Rhonchi,0.156,0.250,8
Stridor,0.148,0.167,6
Wheeze,0.058,0.125,8


## 4. Phase 1: Candidate generation

- **Goal:** almost every artifact that matters should generate some candidate, without an excessive load on the clean signals and without overflowing the artifact.
- **Optimization of:** `WAVELET`, `K`, `ENV_SMOOTH`, `MASK_DILATION`, `REF_INNER`, `REF_OUTER`, `MAX_EVENT`.
- **Output:** C1.

### 4.1. Metrics

Measured apart and with priority (a missed candidate is lost for good; a wider span only degrades its features):

1. **Candidate recall:** proportion of artifacts with SNR ≤ 0 dB that have some candidate in their zone $Z_i$ (artifact ± 25 ms).
2. **Load:** candidates per control signal.
3. **Overflow:** median of $w_i$ over the artifacts that have some candidate:

$$
w_i = \frac{|Z_i|}{|Z_i \cup \text{spans}_i|} = \frac{|Z_i|}{|Z_i| + \text{overflow}_i}
$$

where $\text{spans}_i$ is the union of the candidate spans that touch $Z_i$.

- **Diagnostics:** median overflow (ms) and weighted candidate recall ($\text{Recall}_w = \frac{1}{N}\sum_i w_i$, with $w_i = 0$ if there is no candidate).

### 4.2. Screening

Each parameter is varied alone. If the metrics do not change, it is fixed.

### 4.3. Search

Two Bayesian optimizations, one after the other:

- **(a) Maximum recall:** maximize the recall subject to load < 2 × baseline → recall$_{\max}$.
- **(b) Minimum load:** minimize the load subject to recall ≥ recall$_{\max}$ − 0.005 and median $w_i$ ≥ median $w_i$ of the baseline.

The recall, load and overflow of every evaluated configuration (of both searches) are saved.

### 4.4. Choice

Among all the evaluated configurations (a and b) that meet the two constraints of (b), the one with the minimum load → **C1**.

- **Check:** load–overflow Pareto front of the configurations with recall ≥ recall$_{\max}$ − 0.005 (e.g. whether a little more load would give much less overflow).

### 4.5. Full run

Computation of the estimates needed for the Score in [Phase 3](#6.-Phase-3:-Rule-selection).

## 5. Phase 2: Feature definition

- **Goal:** measure each feature so that it separates the artifacts from the rest as well as possible. The candidates are those of C1 and do not change.
- **Metric:** standardized partial ROC-AUC with a false positive rate ≤ 0.1, within the cases where each feature is used.
- **Output:** candidate table.

### 5.1. Width

- **Features:** `width_med`, `width_max`.
- **Parameters:** `BG_FACTOR`, `WIDTH_FLOOR_DB`, `EXTEND`.

### 5.2. Isolation

- **Feature:** `isolation`.
- **Parameter:** `ISO_WIN`.

### 5.3. Impulsiveness

- **Feature:** `z_max`.
- **Parameters:** `Z_WIN`, `AR_ORDER`.

### 5.4. Choice

Combination with the best mean partial AUC of each group (AUC vs value of each parameter).

### 5.5. Candidate table

Features of every candidate with the chosen parameters.

## 6. Phase 3: Rule selection

- **Goal:** define the rule matrix: 4 cases × 8 features (see [6️⃣ Click rules](README.md#6️⃣-click-rules)).
- **Metric:** $\hat{S}$ (estimated score).
- **Output:** final rule structure and thresholds.

### 6.1. Descriptive analysis

- Distributions of the features per class.
- Pairwise scatter plots per class.
- Correlation matrix.

### 6.2. Univariate analysis

For each case:

1. **Filter** with the main condition of the case.
2. **Minimum condition:** $SE \leq 0.03$ (bootstrap grouped by signal and by artifact); otherwise, relax the filter one percentile and repeat (adaptive margin at the relaxation limit: $|AUC - 0.5| \geq \max(0.1,\ 3 \cdot SE)$).
3. **Prefilter:** AUC of each of the other 7 features → condition ≥ / ≤ / not added ($|AUC - 0.5| \leq 0.1$).

**Output:** AUC matrix (case × feature).

### 6.3. Provisional optimization

- **Goal:** estimate the Score of a rule set from its detections on the candidate table, without running the filter. Same mean as $Q$ ([2.2](#2.2.-Global-quality)), with the quality of each cell $k = (s, a, c)$ split into its detected and undetected parts:

$$
\hat{S} = \frac{1}{|\mathcal{S}| |\mathcal{A}| |\mathcal{C}|}\sum_{s\in\mathcal{S}}\sum_{a\in\mathcal{A}}\sum_{c\in\mathcal{C}}\left[\widehat{\text{Det}}_{s,a,c}\cdot \hat{q}^{\text{det}}_{s,a,c} + \left(1-\widehat{\text{Det}}_{s,a,c}\right)\cdot \hat{q}^{\text{undet}}_{s,a,c}\right] - \lambda \cdot \widehat{FP}
$$

- $\widehat{\text{Det}}_{s,a,c}$: fraction of the $N_{a,c}$ positions of the cell detected by the rules.
- $\hat{q}^{\text{det}}_{s,a,c}$ / $\hat{q}^{\text{undet}}_{s,a,c}$: quality of the detected / undetected positions of the cell used by $\hat{S}$: the values $\bar{q}_k^{\text{det}}$ / $\bar{q}_k^{\text{undet}}$ measured in the full run ([4.5](#4.5.-Full-run)), or imputed when missing ([Imputation](#Imputation)). They do not depend on the rules: only $\widehat{\text{Det}}_{s,a,c}$ and $\widehat{FP}$ change.
- $\widehat{FP}$: number of false positives with the rules, averaged as $FP$.

#### Decomposition of the quality

$\hat{S}$ only uses the cell level; the aggregated means are used for $\lambda$ ([3.2](#3.2.-Penalty-λ)) and the [imputation](#Imputation).

**Cell:** if $p_k$ is the fraction of the positions of cell $k$ detected in the full run, and $\bar{q}_k^{\text{det}}$ and $\bar{q}_k^{\text{undet}}$ are the mean $q$ of its detected and undetected positions:

$$
\bar{q}_k = p_k\cdot \bar{q}_k^{\text{det}} + (1-p_k)\cdot \bar{q}_k^{\text{undet}}
$$

The term in brackets of $\hat{S}$ is this decomposition with $p_k$ replaced by $\widehat{\text{Det}}_{s,a,c}$, keeping $\bar{q}_k^{\text{det}}$ and $\bar{q}_k^{\text{undet}}$ from the full run. With the detections of the full run, $\hat{S} = S$ exactly. With other rules, $\hat{S}$ is an approximation: it assumes that only the fraction of detected positions changes, but the real $q$ of the cell can also change (e.g. the positions that become detected may be harder to attenuate than those already detected). The real $S$ is computed in [Phase 4](#7.-Phase-4:-Attenuation).

**Aggregated means** (per artifact $\bar{q}_{s,a}$, per SNR $\bar{q}_s$): weighted mean of the detected ($\mathcal{D}$) or undetected ($\mathcal{U}$) mixtures, each one with its weight in $Q$:

$$
w_i = \frac{1}{N_{a,c}}, \qquad
\bar{q}^{\text{det}} = \frac{\sum\limits_{i\in\mathcal{D}} w_i\, q_i}{\sum\limits_{i\in\mathcal{D}} w_i}, \qquad
\bar{q}^{\text{undet}} = \frac{\sum\limits_{i\in\mathcal{U}} w_i\, q_i}{\sum\limits_{i\in\mathcal{U}} w_i}
$$

With these weights, the decomposition also holds per SNR, with $\text{Det}_s$ the mean over $a, c$ of $p_k$:

$$
Q_s = \text{Det}_s\cdot \bar{q}^{\text{det}}_s + \left(1-\text{Det}_s\right)\cdot \bar{q}^{\text{undet}}_s
$$

#### Imputation

A value is missing when an artifact is never detected (or never missed) in a cell in the full run, but the new rules do detect (or miss) it. Each value used by $\hat{S}$ is taken from the closest level with data:

- **$\hat{q}^{\text{undet}}_{s,a,c}$:** $\bar{q}_k^{\text{undet}}$; if missing, the filter does not touch the zone of the artifact, so $\text{SNR}^{\text{out}} \approx s$ and $\hat{q}^{\text{undet}}_{s,a,c} \approx q(s)$ (to be checked with the measured values).
- **$\hat{q}^{\text{det}}_{s,a,c}$:** $\bar{q}_k^{\text{det}}$; if missing, $\bar{q}^{\text{det}}_{s,a}$ (same artifact in the other clean signals); if the artifact is not detected in any clean signal at that SNR, $\bar{q}^{\text{det}}_s$.

> If the full run gives $q^{\text{det}}$ of every mixture (forcing the attenuation of its candidate), no imputation is needed.

#### Procedure

1. **Search:** Bayesian optimization over the conditions that passed the prefilter, maximizing $\hat{S}$.
2. **Cross-validation:** 5 folds → 5 sets of thresholds + 5 values of $\hat{S}$.

### 6.4. Importance of each condition

| Measure | Definition |
|:--------|:-----------|
| **Unique rejections** | Number of candidates that meet every condition except this one and that this condition rejects |
| **Ablation** | $\Delta\hat{S}$ when it is disabled |
| **Stability** | Variability of the threshold across folds |
| **Unique detections per case** | Number of artifacts detected by that case and by no other |

**Output:** importance table (initial).

### 6.5. Backward pruning

1. **Pruning:** remove the condition with the highest $\Delta\hat{S}$.
2. **Optimization:** re-optimize the remaining thresholds.
3. **Repeat** until reaching a rule with a single condition.
4. **1-SE rule:** choose the simplest rule set whose $\hat{S}$ is within 1 SE of the best.
5. **Final thresholds:** optimize the thresholds of the remaining conditions on the whole tuning table.

**Outputs:** pruning curve, importance table (final), final rule structure and thresholds.

## 7. Phase 4: Attenuation

- **Optimization of:** `G_FAC`, `GAIN_SMOOTH`, `LOCAL_BG`.
- **Metric:** Score.
- **Output:** the optimized version.

## 8. Phase 5: Validation

- **Metrics:** S, Q, FP.
- **Output:** comparison and generalization.

### 8.1. Sensitivity to λ

Rule selection as a function of the value of $\lambda$.

### 8.2. Comparison

- S (Q, FP) for different values of $\lambda$.
- Wilcoxon signed-rank test, 95 % bootstrap CI of ΔS and number of signals that improve.

### 8.3. Generalization

Optimism = ΔS of the tuning set − ΔS of the test sets (test, new signals and new artifacts).